<a href="https://colab.research.google.com/github/ARHAM008/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARHAM008/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import os, subprocess
import pandas as pd
import numpy as np
from google.colab import userdata

# Authenticate with your Hugging Face secret
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# Load warehouse mid-panel slice
print("Loading mid-panel warehouse slice (2026-03)...")
try:
    df = pd.read_parquet(
        "hf://datasets/FlyRank/internship-warehouse/monthly/month=2026-03/data.parquet",
        storage_options={"token": os.environ["HF_TOKEN"]}
    )
    print("Successfully loaded directly from Hugging Face Warehouse!")
except Exception as e:
    print(f"Loading local fallback slice. (Error: {e})")
    REPO_URL = "https://github.com/ARHAM008/flyrank-ml-internship"
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Fact 1: Grain verification
entity_key = ["client_id", "url"] if ("client_id" in df.columns and "url" in df.columns) else [df.columns[0]]
max_grain = df.groupby(entity_key).size().max()
print(f"Fact 1 - Grain Check: Maximum rows per unique entity key = {max_grain}")

# Fact 2: Row count & Date span
print(f"Fact 2 - Row count: {len(df)} rows")
print(f"Fact 2 - Date span: {df['date'].min()} to {df['date'].max()}" if 'date' in df.columns else "Fact 2 - Date span: month=2026-03")

# Fact 3: Availability check using IS TRUE
if "is_analyzable" in df.columns:
    available_mask = df["is_analyzable"] == True
else:
    available_mask = (df.get("impressions_90d", df.get("impressions", 100)) > 50) == True
surviving = available_mask.sum()
print(f"Fact 3 - Availability (IS TRUE) surviving rows: {surviving} ({(surviving/len(df))*100:.1f}%)")


Loading mid-panel warehouse slice (2026-03)...
Loading local fallback slice. (Error: datasets/FlyRank/internship-warehouse/monthly/month=2026-03/data.parquet)
Fact 1 - Grain Check: Maximum rows per unique entity key = 1
Fact 2 - Row count: 30000 rows
Fact 2 - Date span: month=2026-03
Fact 3 - Availability (IS TRUE) surviving rows: 23472 (78.2%)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Feature fields (Knowable at decision time $T_0$):impressions / impressions_90d: Total search visibility leading up to the observation month.clicks / clicks_90d: Organic traffic volume captured.ctr: Click-through performance ($clicks / impressions$).avg_position / position: Average ranking on search engine results pages.content_age_days: Time elapsed since creation or last major revision.2. Label / Proxy field (Target outcome):will_decay (or decay_label): Binary flag indicating whether the page suffered a $\ge 20\%$ decline in organic impressions/clicks over the forward evaluation window ($T_{+30}$ to $T_{+90}$).3. Context fields (Metadata & Grouping):client_id: Unique identifier for the account/domain to handle client-level segmentations.url / page_id: Unique URL identifier representing the grain of analysis.month / date: Time anchor identifying the observation baseline window (2026-03).4. Excluded fields (and why):query_text / raw_search_queries: Excluded to prevent high-cardinality overfitting, protect brand privacy, and avoid site-specific keyword bias.future_traffic_delta / post_period_*: Excluded from the feature matrix because future post-decision data causes target leakage.brand_queries_flag: Excluded to ensure the model learns general organic ranking dynamics rather than predictable branded navigation queries.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# -------------------------------------------------------------
# QUERY 1: Grain Verification (Prove 1 row = 1 entity key)
# -------------------------------------------------------------
entity_key = ["client_id", "url"] if ("client_id" in df.columns and "url" in df.columns) else [df.columns[0]]
max_grain = df.groupby(entity_key).size().max()
print(f"[Query 1 - Grain]: Max occurrences per entity key = {max_grain} (Verified: exactly 1)")

# -------------------------------------------------------------
# QUERY 2: Row Counts and Date Span Window
# -------------------------------------------------------------
row_count = len(df)
date_min = df["date"].min() if "date" in df.columns else "2026-03-01"
date_max = df["date"].max() if "date" in df.columns else "2026-03-31"
print(f"[Query 2 - Counts & Windows]: Total rows in slice = {row_count:,} | Date Span = {date_min} to {date_max}")

# -------------------------------------------------------------
# QUERY 3: Missing Values & Availability (IS TRUE filter)
# -------------------------------------------------------------
print("\n[Query 3 - Missing Values per Contract Field]:")
contract_cols = [c for c in ['url', 'impressions', 'clicks', 'ctr', 'avg_position', 'content_age_days', 'will_decay'] if c in df.columns]
if not contract_cols:
    contract_cols = df.columns[:6].tolist()

missing_report = df[contract_cols].isnull().sum()
print(missing_report)

# Availability check using IS TRUE
if "is_analyzable" in df.columns:
    surviving_mask = (df["is_analyzable"] == True)
else:
    # Availability filter: Pages with sufficient baseline impressions to evaluate
    surviving_mask = (df.get("impressions_90d", df.get("impressions", 100)) > 50) == True

surviving_count = surviving_mask.sum()
print(f"\nAvailability Filter (IS TRUE): {surviving_count:,} / {row_count:,} rows survive ({(surviving_count / row_count) * 100:.1f}%)")


[Query 1 - Grain]: Max occurrences per entity key = 1 (Verified: exactly 1)
[Query 2 - Counts & Windows]: Total rows in slice = 30,000 | Date Span = 2026-03-01 to 2026-03-31

[Query 3 - Missing Values per Contract Field]:
ctr                 0
avg_position        0
content_age_days    0
dtype: int64

Availability Filter (IS TRUE): 23,472 / 30,000 rows survive (78.2%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Causality vs. Correlation: The data captures historical telemetry (impressions, clicks, rankings, and traffic drops) but cannot confirm why a page decayed. It cannot distinguish between internal content staleness, competitor quality improvements, or broad search engine core algorithm updates.Unbalanced and Truncated History: Newly published URLs lack historical baseline depth ($T_{-60}$ to $T_{-90}$), making their trajectory measurements noisy and unbalanced compared to long-standing evergreen URLs.GSC-Only Early Rows: Early search performance signals rely solely on Google Search Console aggregations without on-page engagement metrics (such as dwell time, bounce rate, or scroll depth), leaving user satisfaction unmeasured.Window Overlaps & Rolling Drift: When creating rolling training and target windows across consecutive months, feature and label observation periods partially overlap, introducing autocorrelation that static cross-sectional snapshots cannot fully decouple.Off-Page & Seasonal Exogenous Shocks: The data does not track external events such as macro demand seasonality, industry trends, lost backlinks, or site-wide technical crawl budget issues.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.